In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SaaS-BigData-Partie2-3")
    .master("local[*]")
    .getOrCreate()
)

In [9]:
spark = (
    SparkSession.builder
    .appName("SaaS-BigData-Partie1")
    .master("spark://spark-master:7077")
    .config("spark.pyspark.python", "/usr/bin/python3.11")
    .config("spark.pyspark.driver.python", "python3")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

In [2]:
import json
import time

sc = spark.sparkContext
t0 = time.time()

# 1. Charger le fichier en RDD et parser chaque ligne (json.loads)
raw_rdd = sc.textFile("/tmp/orders_history.jsonl")
parsed_rdd = raw_rdd.map(lambda line: json.loads(line))

# 2. Garder uniquement les commandes DELIVERED
delivered_rdd = parsed_rdd.filter(lambda ev: ev.get("status") == "DELIVERED")

# 3. Chiffre d'affaires (CA) total par ville avec map + reduceByKey
ca_par_ville_rdd = (
    delivered_rdd
    .map(lambda ev: (ev["city"], float(ev["amount_mad"])))
    .reduceByKey(lambda a, b: round(a + b, 2))
)

# 4. Panier moyen par ville en UN SEUL PASSAGE avec aggregateByKey : (somme, compte)
panier_moyen_rdd = (
    delivered_rdd
    .map(lambda ev: (ev["city"], float(ev["amount_mad"])))
    .aggregateByKey(
        (0.0, 0),
        lambda acc, val: (acc[0] + val, acc[1] + 1),
        lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1]),
    )
    .mapValues(lambda s_c: round(s_c[0] / s_c[1], 2) if s_c[1] > 0 else 0.0)
)

# 5. Top 5 des restaurants par nombre de commandes livrées
top5_restaurants = (
    delivered_rdd
    .map(lambda ev: (ev["restaurant_id"], 1))
    .reduceByKey(lambda a, b: a + b)
    .takeOrdered(5, key=lambda x: -x[1])
)

# Actions pour récupérer et afficher les résultats
ca_dict = dict(ca_par_ville_rdd.collect())
panier_dict = dict(panier_moyen_rdd.collect())

print("=== CA et Panier Moyen par ville (DELIVERED) ===")
print(f"{'Ville':<15} | {'CA Total (MAD)':>15} | {'Panier Moyen (MAD)':>18}")
print("-" * 55)
for city in sorted(ca_dict.keys()):
    print(f"{city:<15} | {ca_dict[city]:>15.2f} | {panier_dict[city]:>18.2f}")

print("\n=== Top 5 Restaurants (nb commandes DELIVERED) ===")
for rank, (resto_id, count) in enumerate(top5_restaurants, start=1):
    print(f"{rank}. {resto_id} : {count} commandes")

t_rdd = time.time() - t0
print(f"\nTemps d'exécution total Partie 2 (RDD) : {t_rdd:.2f} s")

# 6. Lineage (toDebugString)
print("\n=== Lineage (toDebugString) sur panier_moyen_rdd ===")
print(panier_moyen_rdd.toDebugString().decode("utf-8"))

=== CA et Panier Moyen par ville (DELIVERED) ===
Ville           |  CA Total (MAD) | Panier Moyen (MAD)
-------------------------------------------------------
Casablanca      |      1365640.50 |             242.09
Kenitra         |      1328401.83 |             239.83
Marrakech       |      1305473.31 |             245.76
Rabat           |      1325623.75 |             241.99
Tanger          |      1351249.87 |             243.56

=== Top 5 Restaurants (nb commandes DELIVERED) ===
1. R-055 : 177 commandes
2. R-123 : 171 commandes
3. R-165 : 168 commandes
4. R-017 : 166 commandes
5. R-109 : 165 commandes

Temps d'exécution total Partie 2 (RDD) : 15.49 s

=== Lineage (toDebugString) sur panier_moyen_rdd ===
(2) PythonRDD[16] at collect at /tmp/ipykernel_4180/472792246.py:43 []
 |  MapPartitionsRDD[9] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[8] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[7] at aggregateByKey at /tmp/ipykernel_4180/472792246.p

In [3]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

t0_sql = time.time()

# 1. Schémas explicites (StructType) sans inférence
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("restaurant_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("status", StringType(), True),
    StructField("amount_mad", DoubleType(), True),
    StructField("items_count", IntegerType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("courier_id", StringType(), True),
    StructField("event_time", TimestampType(), True),
])

restaurants_schema = StructType([
    StructField("restaurant_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("category", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
])

orders_df = spark.read.schema(orders_schema).json("/tmp/orders_history.jsonl")
restaurants_df = spark.read.schema(restaurants_schema).json("/tmp/restaurants.json")

# 2. CA et panier moyen par ville (DataFrame API + Spark SQL)
delivered_df = orders_df.filter(F.col("status") == "DELIVERED")

ca_ville_df = (
    delivered_df
    .groupBy("city")
    .agg(
        F.round(F.sum("amount_mad"), 2).alias("ca_total_mad"),
        F.round(F.avg("amount_mad"), 2).alias("panier_moyen_mad"),
        F.count("*").alias("nb_commandes"),
    )
    .orderBy("city")
)
print("=== CA et Panier Moyen par ville (DataFrame API) ===")
ca_ville_df.show(truncate=False)

orders_df.createOrReplaceTempView("orders_hist")
restaurants_df.createOrReplaceTempView("restaurants_ref")

ca_ville_sql = spark.sql("""
    SELECT
        city,
        ROUND(SUM(amount_mad), 2) AS ca_total_mad,
        ROUND(AVG(amount_mad), 2) AS panier_moyen_mad,
        COUNT(*) AS nb_commandes
    FROM orders_hist
    WHERE status = 'DELIVERED'
    GROUP BY city
    ORDER BY city
""")
print("=== CA et Panier Moyen par ville (Spark SQL) ===")
ca_ville_sql.show(truncate=False)

# 3. Jointure avec le référentiel restaurants -> CA par catégorie et par ville
joined_df = (
    delivered_df.alias("o")
    .join(restaurants_df.alias("r"), on="restaurant_id", how="inner")
)

ca_cat_ville_df = (
    joined_df
    .groupBy(F.col("o.city").alias("city"), F.col("r.category").alias("category"))
    .agg(
        F.round(F.sum("o.amount_mad"), 2).alias("ca_mad"),
        F.count("*").alias("nb_commandes"),
    )
    .orderBy("city", F.desc("ca_mad"))
)
print("=== CA par catégorie et par ville ===")
ca_cat_ville_df.show(20, truncate=False)

# 4. UDF Python vs expression native when/otherwise
@F.udf(StringType())
def distance_band_udf(dist):
    if dist is None:
        return None
    if dist < 2.0:
        return "court"
    elif dist <= 5.0:
        return "moyen"
    else:
        return "long"

distance_band_native = (
    F.when(F.col("distance_km").isNull(), F.lit(None).cast(StringType()))
    .when(F.col("distance_km") < 2.0, F.lit("court"))
    .when(F.col("distance_km") <= 5.0, F.lit("moyen"))
    .otherwise(F.lit("long"))
)

t_udf_start = time.time()
df_with_udf = orders_df.withColumn("distance_band", distance_band_udf(F.col("distance_km")))
cnt_udf = df_with_udf.groupBy("distance_band").count().collect()
t_udf = time.time() - t_udf_start

t_native_start = time.time()
df_with_native = orders_df.withColumn("distance_band", distance_band_native)
cnt_native = df_with_native.groupBy("distance_band").count().collect()
t_native = time.time() - t_native_start

print(f"Temps UDF Python      : {t_udf:.3f} s -> {[(r['distance_band'], r['count']) for r in cnt_udf]}")
print(f"Temps when/otherwise  : {t_native:.3f} s -> {[(r['distance_band'], r['count']) for r in cnt_native]}")

# 5. Écriture en Parquet partitionné par city
silver_out_path = "/tmp/silver/orders_history"
(
    df_with_native
    .filter(F.col("status") == "DELIVERED")
    .write
    .mode("overwrite")
    .partitionBy("city")
    .parquet(silver_out_path)
)
print(f"Écriture Parquet terminée dans {silver_out_path}")

t_sql_total = time.time() - t0_sql
print(f"Temps total Partie 3 (DataFrames/SQL) : {t_sql_total:.2f} s")

# 6. Plan d'exécution détaillé explain(True) sur la jointure
print("\n=== explain(True) sur la jointure ===")
joined_df.explain(True)

=== CA et Panier Moyen par ville (DataFrame API) ===
+----------+------------+----------------+------------+
|city      |ca_total_mad|panier_moyen_mad|nb_commandes|
+----------+------------+----------------+------------+
|Casablanca|1365640.5   |242.09          |5641        |
|Kenitra   |1328401.83  |239.83          |5539        |
|Marrakech |1305473.31  |245.76          |5312        |
|Rabat     |1325623.75  |241.99          |5478        |
|Tanger    |1351249.87  |243.56          |5548        |
+----------+------------+----------------+------------+

=== CA et Panier Moyen par ville (Spark SQL) ===
+----------+------------+----------------+------------+
|city      |ca_total_mad|panier_moyen_mad|nb_commandes|
+----------+------------+----------------+------------+
|Casablanca|1365640.5   |242.09          |5641        |
|Kenitra   |1328401.83  |239.83          |5539        |
|Marrakech |1305473.31  |245.76          |5312        |
|Rabat     |1325623.75  |241.99          |5478        |
|

In [4]:
parquet_check_df = spark.read.parquet("/tmp/silver/orders_history")
print("Nombre de lignes relues depuis Parquet :", parquet_check_df.count())
parquet_check_df.groupBy("city", "distance_band").count().orderBy("city", "distance_band").show(truncate=False)

Nombre de lignes relues depuis Parquet : 27518
+----------+-------------+-----+
|city      |distance_band|count|
+----------+-------------+-----+
|Casablanca|court        |762  |
|Casablanca|long         |3365 |
|Casablanca|moyen        |1514 |
|Kenitra   |court        |664  |
|Kenitra   |long         |3428 |
|Kenitra   |moyen        |1447 |
|Marrakech |court        |685  |
|Marrakech |long         |3202 |
|Marrakech |moyen        |1425 |
|Rabat     |court        |690  |
|Rabat     |long         |3353 |
|Rabat     |moyen        |1435 |
|Tanger    |court        |735  |
|Tanger    |long         |3381 |
|Tanger    |moyen        |1432 |
+----------+-------------+-----+



In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SaaS-BigData-Partie4-Streaming")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .getOrCreate()
)

print("Spark version :", spark.version)

Spark version : 3.5.1


In [2]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# 1. Recréer la session Spark avec le connecteur Kafka
try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("SaaS-BigData-Partie4-Streaming")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .getOrCreate()
)

kafka_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("restaurant_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("status", StringType(), True),
    StructField("amount_mad", DoubleType(), True),
    StructField("items_count", IntegerType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("courier_id", StringType(), True),
    StructField("event_time", TimestampType(), True),
])

# 2. Lecture du topic Kafka
kafka_bootstrap = "kafka:9092"

raw_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("subscribe", "orders-events")
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

parsed_stream_df = (
    raw_stream_df
    .select(F.from_json(F.col("value").cast("string"), kafka_schema).alias("data"))
    .select("data.*")
    .filter(F.col("order_id").isNotNull())
)

# 3. Agrégation glissante par fenêtre de 1 minute et par ville
windowed_kpi_stream = (
    parsed_stream_df
    .withWatermark("event_time", "2 minutes")
    .filter(F.col("status") == "DELIVERED")
    .groupBy(
        F.window(F.col("event_time"), "1 minute").alias("window"),
        F.col("city").alias("city"),
    )
    .agg(
        F.round(F.sum("amount_mad"), 2).alias("ca_delivered_mad"),
        F.round(F.avg("amount_mad"), 2).alias("panier_moyen_mad"),
        F.count("*").alias("nb_commandes_livrees"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "city",
        "ca_delivered_mad",
        "panier_moyen_mad",
        "nb_commandes_livrees",
    )
)

# 4. Écriture Parquet + affichage console pendant 15 secondes
raw_stream_query = (
    parsed_stream_df.writeStream
    .format("parquet")
    .option("path", "/tmp/streaming/orders_raw_parquet")
    .option("checkpointLocation", "/tmp/checkpoints/orders_raw")
    .outputMode("append")
    .start()
)

kpi_stream_query = (
    windowed_kpi_stream.writeStream
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/checkpoints/orders_kpi_console")
    .outputMode("update")
    .start()
)

time.sleep(15)
kpi_stream_query.stop()
raw_stream_query.stop()
print("Partie 4 (Streaming Kafka) exécutée avec succès !")

Partie 4 (Streaming Kafka) exécutée avec succès !


In [3]:
import urllib.request
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# Rechargement propre des données historiques et du référentiel restaurants
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("restaurant_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("status", StringType(), True),
    StructField("amount_mad", DoubleType(), True),
    StructField("items_count", IntegerType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("courier_id", StringType(), True),
    StructField("event_time", TimestampType(), True),
])

restaurants_schema = StructType([
    StructField("restaurant_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("category", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
])

orders_df = spark.read.schema(orders_schema).json("/tmp/orders_history.jsonl")
restaurants_df = spark.read.schema(restaurants_schema).json("/tmp/restaurants.json")

# Détection SeaweedFS S3 (avec repli local /tmp/lakehouse si besoin)
s3_endpoint = "http://seaweedfs:8333"
use_s3 = False
for candidate in ["http://seaweedfs:8333", "http://localhost:8333"]:
    try:
        urllib.request.urlopen(f"{candidate}/status", timeout=2)
        s3_endpoint = candidate
        use_s3 = True
        break
    except Exception:
        pass

if use_s3:
    spark.conf.set("spark.hadoop.fs.s3a.endpoint", s3_endpoint)
    spark.conf.set("spark.hadoop.fs.s3a.access.key", "any")
    spark.conf.set("spark.hadoop.fs.s3a.secret.key", "any")
    spark.conf.set("spark.hadoop.fs.s3a.path.style.access", "true")
    spark.conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    spark.conf.set("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    base_lake_path = "s3a://bigdata-saas"
    print(f"Stockage cible : SeaweedFS S3 ({s3_endpoint}) -> {base_lake_path}")
else:
    base_lake_path = "/tmp/lakehouse"
    print(f"Stockage cible (repli local) : {base_lake_path}")

# --- BRONZE : Données brutes + métadonnées d'ingestion ---
bronze_orders_df = (
    orders_df
    .withColumn("ingestion_date", F.current_date())
    .withColumn("ingestion_ts", F.current_timestamp())
)

bronze_path = f"{base_lake_path}/bronze/orders"
bronze_orders_df.write.mode("overwrite").parquet(bronze_path)
print(f"Couche BRONZE écrite : {bronze_path} ({bronze_orders_df.count()} lignes)")

# --- SILVER : Nettoyage, tranche de distance & jointure référentiel restaurants ---
silver_clean_df = (
    bronze_orders_df
    .filter(
        F.col("order_id").isNotNull()
        & F.col("amount_mad").isNotNull()
        & (F.col("amount_mad") > 0)
    )
    .withColumn(
        "distance_band",
        F.when(F.col("distance_km") < 2.0, F.lit("court"))
         .when(F.col("distance_km") <= 5.0, F.lit("moyen"))
         .otherwise(F.lit("long")),
    )
    .dropDuplicates(["order_id"])
)

silver_enriched_df = (
    silver_clean_df.alias("o")
    .join(
        restaurants_df.alias("r"),
        F.col("o.restaurant_id") == F.col("r.restaurant_id"),
        "left",
    )
    .select(
        F.col("o.order_id"),
        F.col("o.restaurant_id"),
        F.col("r.name").alias("restaurant_name"),
        F.col("r.category").alias("category"),
        F.col("o.customer_id"),
        F.col("o.city"),
        F.col("o.status"),
        F.col("o.amount_mad"),
        F.col("o.items_count"),
        F.col("o.distance_km"),
        F.col("o.distance_band"),
        F.col("o.courier_id"),
        F.col("o.event_time"),
        F.col("o.ingestion_date"),
    )
)

silver_path = f"{base_lake_path}/silver/orders_enriched"
silver_enriched_df.write.mode("overwrite").partitionBy("city").parquet(silver_path)
print(f"Couche SILVER écrite : {silver_path} ({silver_enriched_df.count()} lignes)")

Stockage cible (repli local) : /tmp/lakehouse
Couche BRONZE écrite : /tmp/lakehouse/bronze/orders (50000 lignes)
Couche SILVER écrite : /tmp/lakehouse/silver/orders_enriched (48973 lignes)


In [4]:
from pyspark.sql import Window
from pyspark.sql import functions as F

silver_df = spark.read.parquet(silver_path)
delivered_silver = silver_df.filter(F.col("status") == "DELIVERED")

# 1. Gold KPI par Ville & Catégorie + classement par CA (Window Function)
gold_city_category = (
    delivered_silver
    .groupBy("city", "category")
    .agg(
        F.count("order_id").alias("nb_commandes_livrees"),
        F.round(F.sum("amount_mad"), 2).alias("ca_total_mad"),
        F.round(F.avg("amount_mad"), 2).alias("panier_moyen_mad"),
        F.round(F.avg("distance_km"), 2).alias("distance_moyenne_km"),
    )
)

win_city = Window.partitionBy("city").orderBy(F.desc("ca_total_mad"))
gold_city_category = gold_city_category.withColumn("rang_ca_ville", F.rank().over(win_city))

gold_city_cat_path = f"{base_lake_path}/gold/kpi_city_category"
gold_city_category.write.mode("overwrite").parquet(gold_city_cat_path)

# 2. Gold Classement des Restaurants
gold_top_restaurants = (
    delivered_silver
    .groupBy("restaurant_id", "restaurant_name", "city", "category")
    .agg(
        F.count("order_id").alias("nb_commandes"),
        F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
        F.round(F.avg("amount_mad"), 2).alias("panier_moyen_mad"),
    )
    .orderBy(F.desc("ca_mad"))
)

gold_resto_path = f"{base_lake_path}/gold/top_restaurants"
gold_top_restaurants.write.mode("overwrite").parquet(gold_resto_path)

# 3. Gold KPI par Tranche de Distance
gold_distance = (
    delivered_silver
    .groupBy("distance_band")
    .agg(
        F.count("order_id").alias("nb_commandes"),
        F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
        F.round(F.avg("amount_mad"), 2).alias("panier_moyen_mad"),
    )
    .orderBy(F.desc("ca_mad"))
)

gold_dist_path = f"{base_lake_path}/gold/kpi_distance_band"
gold_distance.write.mode("overwrite").parquet(gold_dist_path)

# --- Synthèse globale ---
total_orders = silver_df.count()
total_delivered = delivered_silver.count()
taux_livraison = round((total_delivered / total_orders) * 100, 2) if total_orders else 0.0
ca_global = delivered_silver.agg(F.round(F.sum("amount_mad"), 2).alias("ca")).first()["ca"]

print("================ SYNTHÈSE GLOBALE GOLD ================")
print(f"Total commandes traitées     : {total_orders}")
print(f"Commandes livrées (DELIVERED) : {total_delivered} ({taux_livraison}%)")
print(f"Chiffre d'affaires livré     : {ca_global:,.2f} MAD")

print("\n--- Top 10 Restaurants par CA ---")
gold_top_restaurants.show(10, truncate=False)

print("--- KPI par Tranche de Distance ---")
gold_distance.show(truncate=False)

print("--- Top 2 Catégories par Ville ---")
gold_city_category.filter(F.col("rang_ca_ville") <= 2).orderBy("city", "rang_ca_ville").show(truncate=False)

================ SYNTHÈSE GLOBALE GOLD ================
Total commandes traitées     : 48973
Commandes livrées (DELIVERED) : 26948 (55.03%)
Chiffre d'affaires livré     : 6,536,615.02 MAD

--- Top 10 Restaurants par CA ---
+-------------+-------------------+----------+---------+------------+--------+----------------+
|restaurant_id|restaurant_name    |city      |category |nb_commandes|ca_mad  |panier_moyen_mad|
+-------------+-------------------+----------+---------+------------+--------+----------------+
|R-192        |Resto Pizza 192    |Casablanca|pizza    |44          |10826.51|246.06          |
|R-165        |Resto Italien 165  |Casablanca|italien  |42          |10739.66|255.71          |
|R-123        |Resto Sushi 123    |Marrakech |sushi    |36          |10672.28|296.45          |
|R-048        |Resto Tacos 48     |Casablanca|tacos    |40          |10539.52|263.49          |
|R-036        |Resto Burger 36    |Casablanca|burger   |41          |10496.39|256.01          |
|R-149   